# Seequel: Fine-Tuned Text-to-SQL Engine
This notebook follows the blueprint to fine-tune a Qwen2.5 model for generating SQL from natural language.

In [ ]:
import torch
# Check GPU
major_version, minor_version = torch.cuda.get_device_capability()
# Must install unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    # Use this for new GPUs like Ampere, Ada Lovelace, Hopper (A100, H100, L4)
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    # Use this for older GPUs (V100, T4, P100)
    !pip install --no-deps xformers trl peft accelerate bitsandbytes
pass

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-khdtvl1r/unsloth_80858f35c46f4e75aa797d9cc73cbc54
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-khdtvl1r/unsloth_80858f35c46f4e75aa797d9cc73cbc54
  Resolved https://github.com/unslothai/unsloth.git to commit 5b24b8acb51e9f67d009e1d7ba54f9bd01daaf6a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.6 MB/s eta 0:00:00
   ━━

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports RoPE Scaling internally
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0
    bias = "none",    # Optimized to none
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.8.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset

# 1. Load and Shuffle
raw_dataset = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=3407)

# 2. Create proper seeded split
split = raw_dataset.train_test_split(test_size=300, seed=3407)
train_raw = split["train"].select(range(15000))
eval_raw  = split["test"]

SYSTEM_PROMPT = (
    "You are Seequel, an expert SQL generator. "
    "Generate exactly one valid SQLite SELECT query using only the provided schema. "
    "Do not invent tables, columns, values, filters, joins, or conditions. "
    "Return SQL only."
)

def format_example(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Database Schema:\n{example['context']}\n\nQuestion: {example['question']}"},
        ],
        "completion": [
            {"role": "assistant", "content": example["answer"]}
        ],
    }

train_dataset = train_raw.map(format_example, remove_columns=train_raw.column_names)
test_dataset = eval_raw.map(format_example, remove_columns=eval_raw.column_names)

print(f"Training samples: {len(train_dataset)}, Eval samples: {len(test_dataset)}")

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

"<|im_start|>system\nYou are Seequel, an expert SQL generator. Convert the user's natural language question into a valid, optimized SQL query using the provided database schema.<|im_end|>\n<|im_start|>user\nDatabase Schema:\nCREATE TABLE head (age INTEGER)\n\nQuestion: How many heads of the departments are older than 56 ?<|im_end|>\n<|im_start|>assistant\nSELECT COUNT(*) FROM head WHERE age > 56<|im_end|>"

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

TRAIN_MODEL = True # Set to False for inference only
LORA_MODEL_PATH = "seequel_lora_model"

if TRAIN_MODEL:
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = test_dataset,
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        args = SFTConfig(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_ratio = 0.03,
            num_train_epochs = 2,
            learning_rate = 2e-4,
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 10,
            eval_strategy = "steps",
            eval_steps = 100,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
            save_strategy = "steps",
            save_steps = 100,
            save_total_limit = 2,
            load_best_model_at_end = True,
        ),
    )
else:
    print("Inference mode active. Trainer initialization skipped.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/15000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
if TRAIN_MODEL:
    print("Starting training...")
    trainer_stats = trainer.train()

    # Save only after training to avoid overwriting with base model
    model.save_pretrained(LORA_MODEL_PATH)
    tokenizer.save_pretrained(LORA_MODEL_PATH)

    # Export GGUF
    model.save_pretrained_gguf("seequel-1.5b-q4", tokenizer, quantization_method = "q4_k_m")
    print("Training complete and model saved.")
else:
    # Load existing adapters if not training
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = LORA_MODEL_PATH,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)
    print("Adapters loaded for inference.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,000 | Num Epochs = 2 | Total steps = 3,750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.688385
2,2.578951
3,2.638833
4,2.400481
5,2.129683
6,1.919976
7,1.678255
8,1.393958
9,1.062715
10,0.926088


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-900/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_token

### **Saving the Model**
After training, we save the LoRA adapters and export the model to GGUF format for deployment.

In [ ]:
# Save the LoRA adapters (small files)
model.save_pretrained("seequel_lora_model")
tokenizer.save_pretrained("seequel_lora_model")

# Or, save to GGUF for Ollama/Local deployment
# q4_k_m is a good balance between speed and quality
model.save_pretrained_gguf("seequel-1.5b-q4", tokenizer, quantization_method = "q4_k_m")

print("Model saved and exported to GGUF successfully!")

Unsloth: Restored added_tokens_decoder metadata in seequel_lora_model/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in seequel-1.5b-q4/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:52<00:00, 52.20s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:46<00:00, 46.88s/it]


Unsloth: Merge process complete. Saved to `/content/seequel-1.5b-q4`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10241-mix-89aa77b (app-b10241-mix-89aa77b-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['seequel-1.5b-q4_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions complete

### **Generalization Test**
Now that we've fixed the control flow and evaluation logic, let's run a full evaluation on the target test cases to see the true performance.

In [ ]:
# Final Evaluation Run
accuracy = evaluate(verbose=True)
print(f"Final Accuracy Score: {accuracy:.1%}")

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated SQL:
SELECT T1.id, T1.name, T1.salary FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.id WHERE T2.name = 'Engineering' ORDER BY T1.salary DESC LIMIT 3


In [ ]:
# ============================================================
# Seequel Evaluation Suite (Fixed & Expanded)
# ============================================================

TEST_CASES = [
    # ---------- employees / departments ----------
    {
        "schema": """CREATE TABLE employees (id INT, name VARCHAR, salary INT, department_id INT);
CREATE TABLE departments (id INT, name VARCHAR);""",
        "seed": [
            "INSERT INTO departments VALUES (1,'Engineering'),(2,'Sales'),(3,'HR')",
            "INSERT INTO employees VALUES (1,'Alice',150000,1),(2,'Bob',120000,1),(3,'Cara',130000,1),(4,'Dan',90000,2),(5,'Eve',200000,1),(6,'Finn',95000,3)",
        ],
        "question": "Find the top 3 highest paid employees in the Engineering department.",
        "gold_sql": "SELECT id, name, salary FROM employees WHERE department_id = (SELECT id FROM departments WHERE name='Engineering') ORDER BY salary DESC LIMIT 3",
    },
    {
        "schema": """CREATE TABLE employees (id INT, name VARCHAR, salary INT, department_id INT);
CREATE TABLE departments (id INT, name VARCHAR);""",
        "seed": [
            "INSERT INTO departments VALUES (1,'Engineering'),(2,'Sales')",
            "INSERT INTO employees VALUES (1,'Alice',150000,1),(2,'Bob',120000,1),(3,'Dan',90000,2),(4,'Eve',80000,2)",
        ],
        "question": "What is the average salary per department? Show the department name and average salary.",
        "gold_sql": "SELECT d.name, AVG(e.salary) FROM employees e JOIN departments d ON e.department_id = d.id GROUP BY d.name",
    },
    {
        "schema": """CREATE TABLE employees (id INT, name VARCHAR, salary INT, department_id INT);""",
        "seed": ["INSERT INTO employees VALUES (1,'Alice',150000,1),(2,'Bob',120000,1),(3,'Dan',90000,2)"],
        "question": "How many employees are there in total?",
        "gold_sql": "SELECT COUNT(*) FROM employees",
    },
    {
        "schema": """CREATE TABLE employees (id INT, name VARCHAR, salary INT, department_id INT);
CREATE TABLE departments (id INT, name VARCHAR);""",
        "seed": [
            "INSERT INTO departments VALUES (1,'Engineering'),(2,'Sales'),(3,'Legal')",
            "INSERT INTO employees VALUES (1,'Alice',150000,1),(2,'Bob',120000,2)",
        ],
        "question": "List all departments that have no employees.",
        "gold_sql": "SELECT name FROM departments WHERE id NOT IN (SELECT department_id FROM employees)",
    },

    # ---------- ecommerce ----------
    {
        "schema": """CREATE TABLE customers (id INT, name VARCHAR, country VARCHAR);
CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, order_date DATE);""",
        "seed": [
            "INSERT INTO customers VALUES (1,'Ada','US'),(2,'Ben','UK'),(3,'Cy','US')",
            "INSERT INTO orders VALUES (1,1,100.0,'2024-01-05'),(2,1,50.0,'2024-02-01'),(3,2,200.0,'2024-01-20'),(4,3,75.0,'2024-03-01')",
        ],
        "question": "Show the total order amount for each customer, highest first.",
        "gold_sql": "SELECT c.name, SUM(o.amount) AS total FROM customers c JOIN orders o ON c.id=o.customer_id GROUP BY c.name ORDER BY total DESC",
    },
    {
        "schema": """CREATE TABLE customers (id INT, name VARCHAR, country VARCHAR);
CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL);""",
        "seed": [
            "INSERT INTO customers VALUES (1,'Ada','US'),(2,'Ben','UK'),(3,'Cy','US')",
            "INSERT INTO orders VALUES (1,1,100.0),(2,2,200.0)",
        ],
        "question": "Which customers have never placed an order?",
        "gold_sql": "SELECT name FROM customers WHERE id NOT IN (SELECT customer_id FROM orders)",
    },
    {
        "schema": """CREATE TABLE products (id INT, name VARCHAR, category VARCHAR, price DECIMAL, stock INT);""",
        "seed": ["INSERT INTO products VALUES (1,'Pen','Office',1.5,100),(2,'Desk','Furniture',150.0,5),(3,'Chair','Furniture',80.0,0),(4,'Ink','Office',3.0,0)"],
        "question": "List the names of all products that are out of stock.",
        "gold_sql": "SELECT name FROM products WHERE stock = 0",
    },
    {
        "schema": """CREATE TABLE products (id INT, name VARCHAR, category VARCHAR, price DECIMAL, stock INT);""",
        "seed": ["INSERT INTO products VALUES (1,'Pen','Office',1.5,100),(2,'Desk','Furniture',150.0,5),(3,'Chair','Furniture',80.0,10)"],
        "question": "What is the most expensive product in each category?",
        "gold_sql": "SELECT category, MAX(price) FROM products GROUP BY category",
    },
    {
        "schema": """CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, status VARCHAR);""",
        "seed": ["INSERT INTO orders VALUES (1,1,100.0,'shipped'),(2,2,200.0,'pending'),(3,1,50.0,'shipped'),(4,3,300.0,'cancelled')"],
        "question": "What is the total revenue from orders that were shipped?",
        "gold_sql": "SELECT SUM(amount) FROM orders WHERE status='shipped'",
    },

    # ---------- students / courses ----------
    {
        "schema": """CREATE TABLE students (id INT, name VARCHAR, gpa DECIMAL, major VARCHAR);""",
        "seed": ["INSERT INTO students VALUES (1,'Amy',3.9,'CS'),(2,'Bo',3.2,'Math'),(3,'Cy',3.7,'CS'),(4,'Di',2.9,'Math')"],
        "question": "Find students with a GPA above 3.5.",
        "gold_sql": "SELECT name FROM students WHERE gpa > 3.5",
    },
    {
        "schema": """CREATE TABLE students (id INT, name VARCHAR, major VARCHAR);
CREATE TABLE enrollments (student_id INT, course VARCHAR, grade INT);""",
        "seed": [
            "INSERT INTO students VALUES (1,'Amy','CS'),(2,'Bo','Math')",
            "INSERT INTO enrollments VALUES (1,'Algorithms',90),(1,'Databases',85),(2,'Calculus',70)",
        ],
        "question": "What is the average grade for each student? Show the student name.",
        "gold_sql": "SELECT s.name, AVG(e.grade) FROM students s JOIN enrollments e ON s.id=e.student_id GROUP BY s.name",
    },
    {
        "schema": """CREATE TABLE students (id INT, name VARCHAR, major VARCHAR, gpa DECIMAL);""",
        "seed": ["INSERT INTO students VALUES (1,'Amy',3.9,'CS'),(2,'Bo',3.2,'CS'),(3,'Cy',3.7,'Math')"],
        "question": "How many students are enrolled in each major?",
        "gold_sql": "SELECT major, COUNT(*) FROM students GROUP BY major",
    },

    # ---------- date / filtering / distinct ----------
    {
        "schema": """CREATE TABLE events (id INT, name VARCHAR, event_date DATE, attendees INT);""",
        "seed": ["INSERT INTO events VALUES (1,'Kickoff','2024-01-15',30),(2,'Review','2024-06-20',12),(3,'Party','2023-12-31',50)"],
        "question": "List events that happened in 2024.",
        "gold_sql": "SELECT name FROM events WHERE event_date >= '2024-01-01' AND event_date <= '2024-12-31'",
    },
    {
        "schema": """CREATE TABLE employees (id INT, name VARCHAR, department_id INT);""",
        "seed": ["INSERT INTO employees VALUES (1,'A',1),(2,'B',1),(3,'C',2),(4,'D',3)"],
        "question": "How many distinct departments have at least one employee?",
        "gold_sql": "SELECT COUNT(DISTINCT department_id) FROM employees",
    },
    {
        "schema": """CREATE TABLE sales (id INT, region VARCHAR, amount DECIMAL);""",
        "seed": ["INSERT INTO sales VALUES (1,'North',100),(2,'South',200),(3,'North',150),(4,'South',50)"],
        "question": "Show total sales by region, but only regions with total sales over 100.",
        "gold_sql": "SELECT region, SUM(amount) AS total FROM sales GROUP BY region HAVING SUM(amount) > 100",
    },
]

print(f"Loaded {len(TEST_CASES)} corrected test cases.")

Loaded 15 corrected test cases.


In [ ]:
import torch

def generate_sql(schema, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Database Schema:\n{schema}\n\nQuestion: {question}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for assistant role
        return_tensors = "pt",
        return_dict = True,
    ).to("cuda")

    input_length = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        use_cache = True,
        do_sample = False,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.pad_token_id,
    )

    generated_tokens = outputs[0, input_length:]
    sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Clean SQL
    sql = sql.replace("```sql", "").replace("```", "").strip()
    if ";" in sql: sql = sql.split(";")[0].strip()
    return sql

In [ ]:
from collections import Counter

def normalize_value(value):
    if value is None: return None
    if isinstance(value, float): return round(value, 6)
    if isinstance(value, str): return value.strip().lower()
    return value

def normalize_result(rows):
    return [tuple(normalize_value(v) for v in row) for row in rows]

def results_match(gold_rows, gen_rows, ordered=False):
    gold = normalize_result(gold_rows)
    gen = normalize_result(gen_rows)
    if not gold and not gen: return True
    if len(gold) != len(gen): return False

    if ordered:
        return gold == gen
    return Counter(gold) == Counter(gen)

def evaluate(verbose=True):
    pass_count = 0
    valid_exec = 0
    for i, tc in enumerate(TEST_CASES):
        gen_sql = generate_sql(tc["schema"], tc["question"])
        gold_ok, gold_res = run_sql(tc["schema"], tc["seed"], tc["gold_sql"])
        gen_ok, gen_res = run_sql(tc["schema"], tc["seed"], gen_sql)

        is_correct = gold_ok and gen_ok and results_match(gold_res, gen_res, tc.get("ordered", False))

        if is_correct: pass_count += 1
        if gen_ok: valid_exec += 1

        if verbose:
            status = "✅" if is_correct else ("⚠️" if gen_ok else "❌")
            print(f"[{i+1}] {status} Q: {tc['question']}")
            if not is_correct:
                print(f"    SQL: {gen_sql}")
                if not gen_ok: print(f"    Err: {gen_res}")
                else: print(f"    Got: {gen_res} | Want: {gold_res}")

    n = len(TEST_CASES)
    print(f"\nFinal Accuracy: {pass_count}/{n} ({pass_count/n:.1%})")
    return pass_count / n

# Run evaluation
evaluate()

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting evaluation on 15 cases (Deterministic Mode)...



Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] ✅ Q: Find the top 3 highest paid employees in the Engineering department.


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2] ✅ Q: What is the average salary per department? Show the department name and average salary.


... (20 outputs left out)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[13] ⚠️ Q: List events that happened in 2024.
    SQL: SELECT id, name FROM events WHERE event_date LIKE "2024%" AND attendees > 100
    Got: [] | Want: [('Kickoff',), ('Review',)]


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[14] ✅ Q: How many distinct departments have at least one employee?
[15] ✅ Q: Show total sales by region, but only regions with total sales over 100.

Final Accuracy:  7/15 (46.7%)
Exec Validity:   12/15 (80.0%)


In [ ]:
def evaluate_test_set(dataset, num_samples=300):
    print(f"Evaluating on {num_samples} held-out test samples...")
    pass_count = 0
    valid_exec = 0

    for i in range(num_samples):
        sample = dataset[i]
        schema = sample['context']
        question = sample['question']
        gold_sql = sample['answer']

        gen_sql = generate_sql(schema, question)

        # We use empty seed lists for the general test set as they aren't provided
        gold_ok, gold_res = run_sql(schema, [], gold_sql)
        gen_ok, gen_res = run_sql(schema, [], gen_sql)

        # Use the subset match logic to rescue correct data in different column formats
        is_correct = gold_ok and gen_ok and is_subset_match(gold_res, gen_res)

        if is_correct: pass_count += 1
        if gen_ok: valid_exec += 1

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{num_samples}...")

    print("\n" + "="*40)
    print(f"Test Set Accuracy: {pass_count}/{num_samples} ({pass_count/num_samples:.1%})")
    print(f"Test Set Validity: {valid_exec}/{num_samples} ({valid_exec/num_samples:.1%})")
    print("="*40)

# Run this after training to see the generalization score on unseen data
# evaluate_test_set(test_dataset)

In [ ]:
def evaluate_test_set(dataset, num_samples=300):
    print(f"Evaluating on {num_samples} held-out test samples...")
    pass_count = 0
    valid_exec = 0

    for i in range(num_samples):
        sample = dataset[i]
        # Extract schema and question from the source columns
        schema = sample['context']
        question = sample['question']
        gold_sql = sample['answer']

        gen_sql = generate_sql(schema, question)

        # We use empty seed lists for the general test set as they aren't provided
        gold_ok, gold_res = run_sql(schema, [], gold_sql)
        gen_ok, gen_res = run_sql(schema, [], gen_sql)

        is_correct = gold_ok and gen_ok and is_subset_match(gold_res, gen_res)

        if is_correct: pass_count += 1
        if gen_ok: valid_exec += 1

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{num_samples}...")

    print("\n" + "="*40)
    print(f"Test Set Accuracy: {pass_count}/{num_samples} ({pass_count/num_samples:.1%})")
    print(f"Test Set Validity: {valid_exec}/{num_samples} ({valid_exec/num_samples:.1%})")
    print("="*40)

# Run this after training to see generalization score
# evaluate_test_set(test_dataset)